# Web Scraping

# Tarefa 1 - Wikipedia (Steve Jobs)

In [21]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import unquote
import os, time, csv
from datetime import datetime

In [22]:
url_inicial = "https://pt.wikipedia.org/wiki/Steve_Jobs"

requisicao = requests.get(url_inicial, headers={'User-Agent': 'aula-cpa'})
print(requisicao)

soup = BeautifulSoup(requisicao.text, "html.parser")

<Response [200]>


In [23]:
titulo_pagina = soup.head.title.string #titulo da pg
print("Título da página: ", titulo_pagina)

primeiro_paragrafo = soup.find('p') #primeiro paragrafo da pg
print("Primeiro parágrafo: ", primeiro_paragrafo.text)

Título da página:  Steve Jobs – Wikipédia, a enciclopédia livre
Primeiro parágrafo:  Steven Paul Jobs (São Francisco, 24 de fevereiro de 1955 – Palo Alto, 5 de outubro de 2011)[2] foi um inventor, empresário e magnata norte-americano do setor da informática. Notabilizou-se como cofundador, presidente e diretor-executivo da Apple Inc.[6] e por revolucionar seis indústrias: computadores pessoais, filmes de animação, música, telefones, tablets e publicações digitais.[7] Além de sua ligação com a Apple, foi diretor-executivo da empresa de animação por computação gráfica Pixar, e acionista individual máximo da The Walt Disney Company.[8] Morreu no dia 5 de outubro de 2011, aos 56 anos de idade, devido a um câncer pancreático.[2]


In [24]:
links_internos = [] #links internos /wiki/
for a in soup.select("a[href^='/wiki/']"):
    link = "https://pt.wikipedia.org" + a["href"]
    links_internos.append(link)

print("Número de links internos: ", len(links_internos))
print(links_internos[:5])

Número de links internos:  69
['https://pt.wikipedia.org/wiki/Wikip%C3%A9dia:P%C3%A1gina_principal', 'https://pt.wikipedia.org/wiki/Portal:Conte%C3%BAdo_destacado', 'https://pt.wikipedia.org/wiki/Portal:Eventos_atuais', 'https://pt.wikipedia.org/wiki/Wikip%C3%A9dia:Esplanada', 'https://pt.wikipedia.org/wiki/Especial:Aleat%C3%B3ria']


In [25]:
links_imagens = [] #src das imagens
for img in soup.select("img"):
    if img.has_attr("src"):
        links_imagens.append(img["src"])

print("Número de imagens encontradas: ", len(links_imagens))

Número de imagens encontradas:  34


In [26]:
links_corrigidos_imagens = [] #links corrigidos para imagens
for link in links_imagens:
    if link.startswith("//"):
        link = "https:" + link
    elif link.startswith("/"):
        link = "https://pt.wikipedia.org" + link
    links_corrigidos_imagens.append(link)

print("Links corrigidos: ", len(links_corrigidos_imagens))
print(links_corrigidos_imagens[:5])

Links corrigidos:  34
['https://pt.wikipedia.org/static/images/icons/wikipedia.png', 'https://pt.wikipedia.org/static/images/mobile/copyright/wikipedia-wordmark-fr.svg', 'https://pt.wikipedia.org/static/images/mobile/copyright/wikipedia-tagline-pt.svg', 'https://upload.wikimedia.org/wikipedia/commons/thumb/5/51/Steve_Jobs_Headshot_2010_%28cropped_4%29.jpg/250px-Steve_Jobs_Headshot_2010_%28cropped_4%29.jpg', 'https://upload.wikimedia.org/wikipedia/commons/thumb/e/e0/Steve_Jobs_signature.svg/250px-Steve_Jobs_signature.svg.png']


In [27]:
def salvar_html(url, pasta="htmls_lab1"): #pasta onde os arquivos serao salvos
    os.makedirs(pasta, exist_ok=True)
    try:
        resposta = requests.get(url, headers={'User-Agent': 'aula-cpa'})
        resposta.raise_for_status()

        #acentos e caracteres especiais no nome do arquivo
        nome_arquivo = unquote(url.split("/")[-1]) + ".html" 
        caminho = os.path.join(pasta, nome_arquivo)

        with open(caminho, "w", encoding="utf-8") as arquivo:
            arquivo.write(resposta.text)
        return caminho

    except Exception as e:
        print("Erro ao baixar: ", url, "->", e)
        return None

salvar_html(url_inicial)
print("Pagina inicial salva")

Pagina inicial salva


In [28]:
paginas = {url_inicial: requisicao.text} 

for i, url in enumerate(links_internos): 
    try:
        resp = requests.get(url, headers={'User-Agent': 'aula-cpa'}, timeout=10)
        paginas[url] = resp.text
        salvar_html(url)
    except Exception as e:
        print(f"Erro em {url}: {e}")

    if (i + 1) % 20 == 0:
        print(f"{i + 1}/{len(links_internos)} páginas baixadas...")

    time.sleep(0.5)

print(f"Total de paginas baixadas: {len(paginas)}")

20/69 páginas baixadas...
40/69 páginas baixadas...
60/69 páginas baixadas...
Erro ao baixar:  https://pt.wikipedia.org/wiki/Wikip%C3%A9dia:Isen%C3%A7%C3%A3o_de_responsabilidade_geral -> 404 Client Error: Not Found for url: https://pt.wikipedia.org/wiki/Wikip%C3%A9dia:Isen%C3%A7%C3%A3o_de_responsabilidade_geral
Total de paginas baixadas: 66


In [29]:
dados_paginas = []
dados_imagens = []

for url, html in paginas.items():
    s = BeautifulSoup(html, "html.parser")
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    titulo = s.head.title.string if s.head and s.head.title else "N/A"

    p = s.find('p')
    primeiro_paragrafo = p.text if p else "N/A"

    links_pag = ["https://pt.wikipedia.org" + a["href"] for a in s.select("a[href^='/wiki/']")]

    imgs = []
    for img in s.select("img"):
        if img.has_attr("src"):
            src = img["src"]
            if src.startswith("//"):
                src = "https:" + src
            elif src.startswith("/"):
                src = "https://pt.wikipedia.org" + src
            imgs.append(src)
            dados_imagens.append({"pagina_url": url, "titulo": titulo, "imagem_url": src, "timestamp": timestamp})

    dados_paginas.append({
        "url": url,
        "titulo": titulo,
        "primeiro_paragrafo": primeiro_paragrafo,
        "qtd_imagens": len(imgs),
        "links_internos": " | ".join(links_pag),
        "timestamp": timestamp
    })

print(f"Paginas processadas: {len(dados_paginas)}")
print(f"Imagens encontradas: {len(dados_imagens)}")

Paginas processadas: 66
Imagens encontradas: 1173


In [30]:
#CSV dos dados de cada pg
with open("wiki_pages.csv", "w", newline="", encoding="utf-8") as arquivo:
    writer = csv.DictWriter(arquivo, fieldnames=["url", "titulo", "primeiro_paragrafo", "qtd_imagens", "links_internos", "timestamp"])
    writer.writeheader()
    writer.writerows(dados_paginas)

#CSV dos links de imagens
with open("wiki_images.csv", "w", newline="", encoding="utf-8") as arquivo:
    writer = csv.DictWriter(arquivo, fieldnames=["pagina_url", "titulo", "imagem_url", "timestamp"])
    writer.writeheader()
    writer.writerows(dados_imagens)

print("wiki_pages.csv e wiki_images.csv salvos!")

wiki_pages.csv e wiki_images.csv salvos!


In [31]:
for d in dados_paginas[:3]:
    print(f"Título   : {d['titulo']}")
    print(f"URL      : {d['url']}")
    print(f"1º §     : {d['primeiro_paragrafo'][:100]}...")
    print(f"Imagens  : {d['qtd_imagens']} | Timestamp: {d['timestamp']}")
    print("-" * 60)

Título   : Steve Jobs – Wikipédia, a enciclopédia livre
URL      : https://pt.wikipedia.org/wiki/Steve_Jobs
1º §     : Steven Paul Jobs (São Francisco, 24 de fevereiro de 1955 – Palo Alto, 5 de outubro de 2011)[2] foi u...
Imagens  : 34 | Timestamp: 2026-04-10 22:58:14
------------------------------------------------------------
Título   : Wikipédia, a enciclopédia livre
URL      : https://pt.wikipedia.org/wiki/Wikip%C3%A9dia:P%C3%A1gina_principal
1º §     : a enciclopédia livre que todos podem editar....
Imagens  : 29 | Timestamp: 2026-04-10 22:58:14
------------------------------------------------------------
Título   : Portal:Conteúdo destacado – Wikipédia, a enciclopédia livre
URL      : https://pt.wikipedia.org/wiki/Portal:Conte%C3%BAdo_destacado
1º §     : Conteúdo destacado representa o melhor que a Wikipédia tem a oferecer. Esta página faz ligações com ...
Imagens  : 9 | Timestamp: 2026-04-10 22:58:15
------------------------------------------------------------
